# Canonical AME(4,3): baseline vs exact-optimized

Offline-first comparison of exact state equivalence, ideal Bell values, and common all-to-all `u/cz` circuit cost. IQM compilation and raw hardware runs are separate opt-in sections.

In [ ]:
import hashlib
import json
import os
import stat
import sys
from pathlib import Path
from uuid import uuid4

import numpy as np
import pandas as pd
import qiskit.qpy as qpy
from IPython.display import display
from qiskit import transpile
from qiskit.quantum_info import Operator, Statevector


def find_repo_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'qudits_on_qubits').is_dir():
            return candidate
    raise RuntimeError('Cannot find repository root containing pyproject.toml and src/qudits_on_qubits.')


def _checked_iqm_path(path, expected, description):
    path = Path(path)
    try:
        metadata = path.lstat()
    except FileNotFoundError:
        return None
    except OSError as error:
        raise RuntimeError(f'Cannot inspect {description}: {path}') from error
    reparse_attribute = getattr(stat, 'FILE_ATTRIBUTE_REPARSE_POINT', 0x400)
    if stat.S_ISLNK(metadata.st_mode) or bool(getattr(metadata, 'st_file_attributes', 0) & reparse_attribute):
        raise RuntimeError(f'Refusing symlink or reparse {description}: {path}')
    if expected == 'file' and not stat.S_ISREG(metadata.st_mode):
        raise RuntimeError(f'Malformed Git metadata or IQM .env candidate: expected file at {path}')
    if expected == 'dir' and not stat.S_ISDIR(metadata.st_mode):
        raise RuntimeError(f'Malformed Git metadata: expected directory at {path}')
    return path.resolve()


def resolve_iqm_env_path(repo_root):
    repo_root = Path(repo_root)
    local_env = _checked_iqm_path(repo_root / '.env', 'file', 'IQM .env candidate')
    if local_env is not None:
        return local_env
    git_metadata = repo_root / '.git'
    metadata = _checked_iqm_path(git_metadata, 'any', 'Git metadata')
    if metadata is None:
        raise RuntimeError(f'Missing Git metadata; cannot resolve IQM .env for {repo_root}')
    owning_repo = repo_root.resolve()
    if stat.S_ISREG(git_metadata.lstat().st_mode):
        try:
            gitdir_line = metadata.read_text(encoding='utf-8').strip()
        except OSError as error:
            raise RuntimeError(f'Malformed Git metadata: cannot read {git_metadata}') from error
        if not gitdir_line.lower().startswith('gitdir:'):
            raise RuntimeError(f'Malformed Git metadata: expected gitdir in {git_metadata}')
        gitdir_value = gitdir_line[7:].strip()
        if not gitdir_value:
            raise RuntimeError(f'Malformed Git metadata: empty gitdir in {git_metadata}')
        gitdir = Path(gitdir_value)
        if not gitdir.is_absolute():
            gitdir = metadata.parent / gitdir
        gitdir = _checked_iqm_path(gitdir, 'dir', 'Git worktree metadata')
        if gitdir is None:
            raise RuntimeError(f'Malformed Git metadata: missing worktree directory for {git_metadata}')
        commondir = _checked_iqm_path(gitdir / 'commondir', 'file', 'Git commondir metadata')
        if commondir is None:
            raise RuntimeError(f'Malformed Git metadata: missing commondir in {gitdir}')
        try:
            commondir_value = commondir.read_text(encoding='utf-8').strip()
        except OSError as error:
            raise RuntimeError(f'Malformed Git metadata: cannot read {commondir}') from error
        if not commondir_value:
            raise RuntimeError(f'Malformed Git metadata: empty commondir in {commondir}')
        common_git_dir = Path(commondir_value)
        if not common_git_dir.is_absolute():
            common_git_dir = gitdir / common_git_dir
        common_git_dir = _checked_iqm_path(common_git_dir, 'dir', 'Git common directory')
        if common_git_dir is None:
            raise RuntimeError(f'Malformed Git metadata: missing common directory for {gitdir}')
        if common_git_dir.name != '.git':
            raise RuntimeError('Cannot use non-.git common directory for IQM .env fallback; provide checkout-local .env or explicit env_path.')
        owner_git_dir = _checked_iqm_path(common_git_dir.parent / '.git', 'dir', 'Git common directory')
        expected_worktrees_dir = _checked_iqm_path(common_git_dir / 'worktrees', 'dir', 'Git worktrees directory')
        if owner_git_dir != common_git_dir or expected_worktrees_dir is None or gitdir.parent != expected_worktrees_dir:
            raise RuntimeError('Cannot validate non-bare owning repository for IQM .env fallback; provide checkout-local .env or explicit env_path.')
        owning_repo = _checked_iqm_path(common_git_dir.parent, 'dir', 'owning repository')
        if owning_repo is None:
            raise RuntimeError('Cannot validate owning repository for IQM .env fallback; provide checkout-local .env or explicit env_path.')
        ownership_error = 'Cannot validate worktree ownership for IQM .env fallback; provide checkout-local .env or explicit env_path.'
        try:
            backlink = _checked_iqm_path(gitdir / 'gitdir', 'file', 'Git worktree backlink')
        except (OSError, RuntimeError):
            raise RuntimeError(ownership_error) from None
        if backlink is None:
            raise RuntimeError(ownership_error)
        try:
            backlink_value = backlink.read_text(encoding='utf-8').strip()
        except (OSError, UnicodeError):
            raise RuntimeError(ownership_error) from None
        if not backlink_value:
            raise RuntimeError(ownership_error)
        try:
            backlink_checkout = Path(backlink_value)
            if not backlink_checkout.is_absolute():
                backlink_checkout = gitdir / backlink_checkout
            backlink_checkout = backlink_checkout.resolve(strict=True)
        except (OSError, RuntimeError, ValueError):
            raise RuntimeError(ownership_error) from None
        if backlink_checkout != metadata:
            raise RuntimeError(ownership_error)
    candidates = [repo_root / '.env']
    if owning_repo != repo_root.resolve():
        candidates.append(owning_repo / '.env')
    for candidate in candidates:
        checked = _checked_iqm_path(candidate, 'file', 'IQM .env candidate')
        if checked is not None:
            return checked
    raise RuntimeError(f'Cannot find IQM .env file for checkout or owning repository: {repo_root}')


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qudits_on_qubits.benchmarks.direct_basis.circuits import (
    build_direct_basis_graph_state_circuit,
    build_exact_optimized_direct_basis_graph_state_circuit,
)
from qudits_on_qubits.experiments import (
    AerIdeal, BootstrapConfig, ExperimentSpec, IQMHardware, MitigationConfig,
    PathBasis, TranspilationConfig, create_backend_adapter, run_experiment,
)
from qudits_on_qubits.experiments.artifacts import load_basis_artifacts
from qudits_on_qubits.experiments.preparation import prepare_measurements
from qudits_on_qubits.reference_experiments import get_encoding, get_reference_experiment


## Validated comparison bundles

In [ ]:
BUNDLE_NAMES = {
    'baseline': 'canonical_ez_comparison_baseline',
    'exact_optimized': 'canonical_ez_comparison_exact_optimized',
}
REQUIRED_BUNDLE_FILES = {'graph_state_direct_basis.qpy', 'E.npy', 'metadata.json'}


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def is_symlink_or_reparse(path):
    metadata = Path(path).lstat()
    reparse_attribute = getattr(stat, 'FILE_ATTRIBUTE_REPARSE_POINT', 0x400)
    return stat.S_ISLNK(metadata.st_mode) or bool(getattr(metadata, 'st_file_attributes', 0) & reparse_attribute)


def assert_safe_path_components(repo_root, path):
    repo_root = Path(repo_root)
    path = Path(path)
    try:
        relative_parts = path.relative_to(repo_root).parts
        resolved_root = repo_root.resolve(strict=True)
    except (OSError, ValueError) as error:
        raise RuntimeError(f'comparison bundle path is outside an available repository root: {path}') from error
    current = repo_root
    for part in ('', *relative_parts):
        if part:
            current = current / part
        try:
            current.lstat()
        except FileNotFoundError:
            continue
        except OSError as error:
            raise RuntimeError(f'unable to inspect comparison bundle path: {current}') from error
        if is_symlink_or_reparse(current):
            raise RuntimeError('comparison bundle path must not contain a symlink or reparse point')
        if not current.resolve(strict=True).is_relative_to(resolved_root):
            raise RuntimeError('comparison bundle path escapes the repository root')


def ensure_safe_directory(repo_root, directory):
    repo_root = Path(repo_root)
    directory = Path(directory)
    assert_safe_path_components(repo_root, directory)
    current = repo_root
    for part in directory.relative_to(repo_root).parts:
        current = current / part
        if not current.exists():
            current.mkdir()
        assert_safe_path_components(repo_root, current)
        if not current.is_dir():
            raise RuntimeError(f'comparison bundle path component must be a directory: {current}')


def load_single_circuit(path):
    try:
        with Path(path).open('rb') as handle:
            circuits = qpy.load(handle)
    except Exception as error:
        raise RuntimeError(f'unable to load comparison QPY: {path}') from error
    if len(circuits) != 1:
        raise RuntimeError('comparison QPY must contain exactly one circuit')
    return circuits[0]


def state_fidelity(first, second):
    first_state = Statevector.from_instruction(first)
    second_state = Statevector.from_instruction(second)
    return float(abs(np.vdot(first_state.data, second_state.data)) ** 2)


def _same_instruction_structure(actual_circuit, expected_circuit):
    if len(actual_circuit.data) != len(expected_circuit.data):
        return False
    for actual, expected in zip(actual_circuit.data, expected_circuit.data):
        actual_qargs = tuple(actual_circuit.find_bit(bit).index for bit in actual.qubits)
        expected_qargs = tuple(expected_circuit.find_bit(bit).index for bit in expected.qubits)
        if type(actual.operation) is not type(expected.operation) or actual_qargs != expected_qargs:
            return False
        try:
            if not np.allclose(Operator(actual.operation).data, Operator(expected.operation).data, atol=1e-12, rtol=0):
                return False
        except Exception:
            return False
    return True


def validate_comparison_bundle(directory, variant, expected_encoding, expected_circuit):
    directory = Path(directory)
    assert_safe_path_components(directory.parents[3], directory)
    if {path.name for path in directory.iterdir()} != REQUIRED_BUNDLE_FILES:
        raise RuntimeError(f'{variant} comparison bundle has unexpected files')
    encoding_path = directory / 'E.npy'
    circuit_path = directory / 'graph_state_direct_basis.qpy'
    try:
        actual_encoding = np.load(encoding_path, allow_pickle=False)
    except Exception as error:
        raise RuntimeError(f'{variant} comparison encoding is invalid') from error
    if actual_encoding.shape != (4, 3) or not np.array_equal(actual_encoding, expected_encoding):
        raise RuntimeError(f'{variant} comparison encoding does not match canonical_ez')
    actual_circuit = load_single_circuit(circuit_path)
    if actual_circuit.num_qubits != 8 or actual_circuit.num_clbits != 0:
        raise RuntimeError(f'{variant} comparison circuit must be an unmeasured eight-qubit circuit')
    if not Statevector.from_instruction(actual_circuit).equiv(Statevector.from_instruction(expected_circuit)):
        raise RuntimeError(f'{variant} comparison circuit prepares the wrong state')
    if not _same_instruction_structure(actual_circuit, expected_circuit):
        raise RuntimeError(f'{variant} comparison circuit structure is stale or deoptimized')
    expected_metadata = {
        'schema': 'qoq-reference-basis-v1', 'state': 'ame43', 'encoding_id': 'canonical_ez',
        'variant': variant, 'num_qubits': 8, 'encoding_shape': [4, 3],
        'files': {
            'graph_state_direct_basis.qpy': {'sha256': sha256_file(circuit_path)},
            'E.npy': {'sha256': sha256_file(encoding_path)},
        },
    }
    try:
        metadata = json.loads((directory / 'metadata.json').read_text(encoding='utf-8'))
    except Exception as error:
        raise RuntimeError(f'{variant} comparison metadata is invalid') from error
    if metadata != expected_metadata:
        raise RuntimeError(f'{variant} comparison metadata does not match validated files')


def _cleanup_staging(repo_root, staging):
    staging = Path(staging)
    assert_safe_path_components(repo_root, staging)
    if not staging.exists():
        return
    if not staging.name.startswith('.canonical_ez_comparison_') or '.tmp-' not in staging.name:
        raise RuntimeError('refusing to clean an unexpected comparison staging directory')
    for name in REQUIRED_BUNDLE_FILES:
        path = staging / name
        if path.exists():
            if is_symlink_or_reparse(path):
                raise RuntimeError('comparison staging file must not be a symlink or reparse point')
            path.unlink()
    staging.rmdir()


def _materialize_comparison_bundle(repo_root, variant, encoding, circuit):
    repo_root = Path(repo_root)
    parent = repo_root / 'experiment_inputs' / 'reference_bases' / 'ame43'
    directory = parent / BUNDLE_NAMES[variant]
    ensure_safe_directory(repo_root, parent)
    assert_safe_path_components(repo_root, directory)
    if directory.exists():
        validate_comparison_bundle(directory, variant, encoding, circuit)
        return directory
    staging = parent / f'.{BUNDLE_NAMES[variant]}.tmp-{uuid4().hex}'
    staging.mkdir()
    try:
        qpy_path = staging / 'graph_state_direct_basis.qpy'
        encoding_path = staging / 'E.npy'
        with qpy_path.open('wb') as handle:
            qpy.dump(circuit, handle)
        with encoding_path.open('wb') as handle:
            np.save(handle, encoding, allow_pickle=False)
        metadata = {
            'schema': 'qoq-reference-basis-v1', 'state': 'ame43', 'encoding_id': 'canonical_ez',
            'variant': variant, 'num_qubits': 8, 'encoding_shape': [4, 3],
            'files': {
                'graph_state_direct_basis.qpy': {'sha256': sha256_file(qpy_path)},
                'E.npy': {'sha256': sha256_file(encoding_path)},
            },
        }
        (staging / 'metadata.json').write_text(json.dumps(metadata, indent=2, sort_keys=True) + '\n', encoding='utf-8')
        validate_comparison_bundle(staging, variant, encoding, circuit)
        try:
            staging.rename(directory)
        except FileExistsError:
            validate_comparison_bundle(directory, variant, encoding, circuit)
        return directory
    finally:
        _cleanup_staging(repo_root, staging)


def build_comparison_circuits():
    encoding = get_encoding('canonical_ez').as_array()
    return {
        'baseline': build_direct_basis_graph_state_circuit('ame43', encoding),
        'exact_optimized': build_exact_optimized_direct_basis_graph_state_circuit('ame43', encoding),
    }


def materialize_comparison_bundles(repo_root):
    encoding = get_encoding('canonical_ez').as_array()
    circuits = build_comparison_circuits()
    return {
        variant: _materialize_comparison_bundle(repo_root, variant, encoding, circuit)
        for variant, circuit in circuits.items()
    }


In [ ]:
OFFLINE_SHOTS = 512
HARDWARE_SHOTS = 50
IQM_SEED = 13
OFFLINE_UNCERTAINTY = BootstrapConfig(samples=100, seed=7)
OFFLINE_TRANSPILATION = TranspilationConfig(optimization_level=3, seed_transpiler=0)
IQM_TRANSPILATION = TranspilationConfig(optimization_level=3, seed_transpiler=IQM_SEED)
RAW_HARDWARE_MITIGATION = MitigationConfig(readout=False, zne=False)
REFERENCE = get_reference_experiment('ame43')
RUN_IQM_COMPILE = False
RUN_IQM_HARDWARE = False


In [ ]:
CIRCUITS = build_comparison_circuits()
BUNDLES = materialize_comparison_bundles(REPO_ROOT)
STATE_FIDELITY = state_fidelity(CIRCUITS['baseline'], CIRCUITS['exact_optimized'])
assert Statevector.from_instruction(CIRCUITS['baseline']).equiv(Statevector.from_instruction(CIRCUITS['exact_optimized']))
assert np.isclose(STATE_FIDELITY, 1.0, atol=1e-12, rtol=0)
BUNDLES, STATE_FIDELITY


## Common all-to-all `u/cz` compiler metrics

In [ ]:
def compile_all_to_all_u_cz(circuit):
    return transpile(circuit, basis_gates=['u', 'cz'], optimization_level=3, seed_transpiler=0)


def collect_circuit_metrics(bundles, circuits):
    rows = {}
    for variant in ('baseline', 'exact_optimized'):
        artifacts = load_basis_artifacts(PathBasis(bundles[variant]), 'ame43')
        measured = prepare_measurements(artifacts).circuits
        if len(measured) != 13:
            raise RuntimeError(f'AME43 comparison expected 13 measurement circuits, found {len(measured)}')
        native_preparation = compile_all_to_all_u_cz(circuits[variant])
        native_measurements = tuple(compile_all_to_all_u_cz(circuit) for circuit in measured)
        measurement_cz = [circuit.count_ops().get('cz', 0) for circuit in native_measurements]
        measurement_depth = [circuit.depth() for circuit in native_measurements]
        rows[variant] = {
            'variant': variant,
            'logical_depth': circuits[variant].depth(),
            'preparation_cz': native_preparation.count_ops().get('cz', 0),
            'preparation_depth': native_preparation.depth(),
            'measurement_circuit_count': len(native_measurements),
            'measurement_cz_total': sum(measurement_cz),
            'measurement_cz_min': min(measurement_cz),
            'measurement_cz_max': max(measurement_cz),
            'measurement_depth_total': sum(measurement_depth),
            'measurement_depth_min': min(measurement_depth),
            'measurement_depth_max': max(measurement_depth),
        }
    if rows['exact_optimized']['preparation_cz'] >= rows['baseline']['preparation_cz']:
        raise RuntimeError('exact-optimized preparation did not reduce all-to-all CZ count')
    if rows['exact_optimized']['preparation_depth'] >= rows['baseline']['preparation_depth']:
        raise RuntimeError('exact-optimized preparation did not reduce all-to-all depth')
    return rows


## Shared-seed Aer comparison

In [ ]:
def run_aer_comparison(bundles, repo_root, shots=OFFLINE_SHOTS):
    results = {}
    for variant in ('baseline', 'exact_optimized'):
        spec = ExperimentSpec(
            state='ame43',
            basis=PathBasis(bundles[variant]),
            backend=AerIdeal(seed_simulator=11),
            shots=shots,
            uncertainty=OFFLINE_UNCERTAINTY,
            transpilation=OFFLINE_TRANSPILATION,
            output_root=Path(repo_root) / 'artifacts' / 'ame43_comparison_runs',
            tags={'baseline': 'canonical_ez', 'preparation': variant, 'backend': 'aer_ideal'},
        )
        results[variant] = run_experiment(spec, repo_root=repo_root)
    return results


AER_RESULTS = run_aer_comparison(BUNDLES, REPO_ROOT)
CIRCUIT_METRICS = collect_circuit_metrics(BUNDLES, CIRCUITS)
COMPARISON_ROWS = []
for variant in ('baseline', 'exact_optimized'):
    estimate = AER_RESULTS[variant].values['raw']['estimate']['real']
    COMPARISON_ROWS.append({
        **CIRCUIT_METRICS[variant],
        'aer_bell_estimate': estimate,
        'classical_bound': REFERENCE.bell_functional.classical_bound,
        'ideal_bell_value': REFERENCE.expected.ideal_bell_value,
        'state_fidelity': STATE_FIDELITY,
    })
COMPARISON_TABLE = pd.DataFrame(COMPARISON_ROWS).set_index('variant')


In [ ]:
display(COMPARISON_TABLE)

## Optional read-only IQM compilation

This resolves Garnet and compiles both 13-circuit batches. It does not submit a job.

In [ ]:
if RUN_IQM_COMPILE:
    IQM_ENV_PATH = resolve_iqm_env_path(REPO_ROOT)
    IQM_ADAPTER = create_backend_adapter(IQMHardware(device='garnet', use_metrics=True, env_path=IQM_ENV_PATH))
    IQM_COMPILE_METRICS = {}
    for variant in ('baseline', 'exact_optimized'):
        artifacts = load_basis_artifacts(PathBasis(BUNDLES[variant]), 'ame43')
        measured = prepare_measurements(artifacts).circuits
        compiled = IQM_ADAPTER.compile(measured, IQM_TRANSPILATION).circuits
        cz_counts = [circuit.count_ops().get('cz', 0) for circuit in compiled]
        depths = [circuit.depth() for circuit in compiled]
        IQM_COMPILE_METRICS[variant] = {
            'circuit_count': len(compiled), 'cz_total': sum(cz_counts),
            'cz_min': min(cz_counts), 'cz_max': max(cz_counts),
            'depth_total': sum(depths), 'depth_min': min(depths), 'depth_max': max(depths),
        }
    display(pd.DataFrame.from_dict(IQM_COMPILE_METRICS, orient='index'))
else:
    print('IQM read-only compilation skipped; set RUN_IQM_COMPILE = True to compile.')


## Optional raw IQM hardware comparison

Both runs use 50 shots, seed 13, and no readout mitigation or ZNE.

In [ ]:
if RUN_IQM_HARDWARE:
    IQM_ENV_PATH = resolve_iqm_env_path(REPO_ROOT)
    BASELINE_IQM_SPEC = ExperimentSpec(
        state='ame43', basis=PathBasis(BUNDLES['baseline']),
        backend=IQMHardware(device='garnet', use_metrics=True, env_path=IQM_ENV_PATH),
        shots=HARDWARE_SHOTS, mitigation=RAW_HARDWARE_MITIGATION,
        uncertainty=OFFLINE_UNCERTAINTY, transpilation=IQM_TRANSPILATION,
        output_root=REPO_ROOT / 'artifacts' / 'ame43_comparison_runs',
        tags={'baseline': 'canonical_ez', 'preparation': 'baseline', 'backend': 'iqm_garnet_raw'},
    )
    OPTIMIZED_IQM_SPEC = ExperimentSpec(
        state='ame43', basis=PathBasis(BUNDLES['exact_optimized']),
        backend=IQMHardware(device='garnet', use_metrics=True, env_path=IQM_ENV_PATH),
        shots=HARDWARE_SHOTS, mitigation=RAW_HARDWARE_MITIGATION,
        uncertainty=OFFLINE_UNCERTAINTY, transpilation=IQM_TRANSPILATION,
        output_root=REPO_ROOT / 'artifacts' / 'ame43_comparison_runs',
        tags={'baseline': 'canonical_ez', 'preparation': 'exact_optimized', 'backend': 'iqm_garnet_raw'},
    )
    IQM_HARDWARE_RESULTS = {
        'baseline': run_experiment(BASELINE_IQM_SPEC, repo_root=REPO_ROOT),
        'exact_optimized': run_experiment(OPTIMIZED_IQM_SPEC, repo_root=REPO_ROOT),
    }
else:
    print('IQM hardware skipped; set RUN_IQM_HARDWARE = True to submit two raw jobs.')
